In [30]:
import netCDF4 as nc
import os
import numpy as np
from config import DATA_DIR  # Importing DATA_DIR from your config file

# Define the path to the test data directory
TEST_DATASET_DIR = os.path.join(DATA_DIR, 'test_data')

# Ensure the test data directory exists
os.makedirs(TEST_DATASET_DIR, exist_ok=True)

def create_target_grid(grid_filename, lat_resolution=0.25, lon_resolution=0.25):
    """Creates a target grid NetCDF file with 0.25-degree resolution."""
    lat = np.arange(-90, 90 + lat_resolution, lat_resolution)
    lon = np.arange(-180, 180 + lon_resolution, lon_resolution)

    # Create a NetCDF file to store the grid
    with nc.Dataset(grid_filename, 'w', format='NETCDF4') as grid_file:
        # Create dimensions
        grid_file.createDimension('latitude', len(lat))
        grid_file.createDimension('longitude', len(lon))

        # Create variables
        latitudes = grid_file.createVariable('latitude', 'f4', ('latitude',))
        longitudes = grid_file.createVariable('longitude', 'f4', ('longitude',))

        # Assign values
        latitudes[:] = lat
        longitudes[:] = lon

        # Add metadata
        latitudes.units = 'degrees_north'
        latitudes.long_name = 'Latitude'
        longitudes.units = 'degrees_east'
        longitudes.long_name = 'Longitude'

    print(f"Target grid created: {grid_filename}")

def create_sine_wave_data(output_filename, year=2003, lat_resolution=0.25, lon_resolution=0.25):
    """Creates synthetic sine wave fire PM2.5 data for testing with 0.25-degree resolution."""
    # Define time steps for daily data (365 days)
    time_steps = 365

    # Define lat and lon ranges
    lat = np.arange(-90, 90 + lat_resolution, lat_resolution)
    lon = np.arange(-180, 180 + lon_resolution, lon_resolution)

    # Generate sine wave pattern for time series (range [-1, 1])
    data = np.sin(np.linspace(0, 2 * np.pi, time_steps))[:, None, None] * np.ones((1, len(lat), len(lon)))

    # Create the NetCDF file
    with nc.Dataset(output_filename, 'w', format='NETCDF4') as nc_file:
        # Create dimensions
        nc_file.createDimension('time', time_steps)
        nc_file.createDimension('latitude', len(lat))
        nc_file.createDimension('longitude', len(lon))

        # Create variables
        times = nc_file.createVariable('time', 'f4', ('time',))
        latitudes = nc_file.createVariable('latitude', 'f4', ('latitude',))
        longitudes = nc_file.createVariable('longitude', 'f4', ('longitude',))
        fire_data = nc_file.createVariable('fire_pm25', 'f4', ('time', 'latitude', 'longitude'))

        # Assign values
        times[:] = np.arange(time_steps)
        latitudes[:] = lat
        longitudes[:] = lon
        fire_data[:] = data

        # Add metadata
        latitudes.units = 'degrees_north'
        latitudes.long_name = 'Latitude'
        longitudes.units = 'degrees_east'
        longitudes.long_name = 'Longitude'
        fire_data.units = 'ug/m^3'
        fire_data.long_name = f"Synthetic Sine Wave PM2.5 for {year}"

    print(f"Sine wave PM2.5 data created: {output_filename}")

def create_test_data():
    """Generates target grid and synthetic sine wave data, both on 0.25-degree grid."""
    # Define filenames for the target grid and test data
    target_grid_file = os.path.join(TEST_DATASET_DIR, 'target_grid_0.25.nc')
    years = [2003, 2004]  # Example years

    # Create target grid
    create_target_grid(target_grid_file)

    # Create synthetic sine wave data for each year, following the correct naming convention
    for year in years:
        fire_data_file = os.path.join(TEST_DATASET_DIR, f'globPMfire02deg_{year}.nc4')
        create_sine_wave_data(fire_data_file, year=year)

def inspect_netcdf_file(file_path):
    """Function to open and inspect a NetCDF file."""
    print(f"\nOpening file: {file_path}")
    
    # Open the NetCDF dataset
    ds = nc.Dataset(file_path, 'r')
    
    # Print the dimensions in the dataset
    print("\nDimensions:")
    for dim in ds.dimensions.values():
        print(f"{dim.name}: {dim.size}")
    
    # Print the variables in the dataset
    print("\nVariables:")
    for var in ds.variables.values():
        print(f"{var.name}: {var.dimensions}, {var.shape}")
    
    # Load some data samples to inspect (if applicable)
    if 'latitude' in ds.variables:
        lat = ds.variables['latitude'][:]
        print("\nLatitude values:", lat[:5], "...")  # Show the first 5 latitudes

    if 'longitude' in ds.variables:
        lon = ds.variables['longitude'][:]
        print("Longitude values:", lon[:5], "...")   # Show the first 5 longitudes

    if 'time' in ds.variables:
        time = ds.variables['time'][:]
        print("Time values:", time[:5], "...")  # Show the first 5 time steps

    if 'fire_pm25' in ds.variables:
        fire_pm25 = ds.variables['fire_pm25']
        print("\nFire PM2.5 data sample (first time step):")
        print(fire_pm25[0, :, :])  # Show the data for the first time step
    
    # Close the dataset
    ds.close()

if __name__ == '__main__':
    # Create all test data
    create_test_data()

    # Paths to the created files
    test_data_file = os.path.join(TEST_DATASET_DIR, 'globPMfire02deg_2003.nc4')
    target_grid_file = os.path.join(TEST_DATASET_DIR, 'target_grid_0.25.nc')

    # Inspect both the test data file and the target grid file
    inspect_netcdf_file(test_data_file)
    inspect_netcdf_file(target_grid_file)


Target grid created: /Users/szelie/OneDrive - ETH Zurich/data/health_multi_risk_data/test_data/target_grid_0.25.nc
Sine wave PM2.5 data created: /Users/szelie/OneDrive - ETH Zurich/data/health_multi_risk_data/test_data/globPMfire02deg_2003.nc4
Sine wave PM2.5 data created: /Users/szelie/OneDrive - ETH Zurich/data/health_multi_risk_data/test_data/globPMfire02deg_2004.nc4

Opening file: /Users/szelie/OneDrive - ETH Zurich/data/health_multi_risk_data/test_data/globPMfire02deg_2003.nc4

Dimensions:
time: 365
latitude: 721
longitude: 1441

Variables:
time: ('time',), (365,)
latitude: ('latitude',), (721,)
longitude: ('longitude',), (1441,)
fire_pm25: ('time', 'latitude', 'longitude'), (365, 721, 1441)

Latitude values: [-90.   -89.75 -89.5  -89.25 -89.  ] ...
Longitude values: [-180.   -179.75 -179.5  -179.25 -179.  ] ...
Time values: [0. 1. 2. 3. 4.] ...

Fire PM2.5 data sample (first time step):
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ...